# THOR single-sequence variant generation (Colab)

<img src="https://raw.githubusercontent.com/Umbriona/THOR/main/Covver_image_small.png" height="200" align="right" style="height:240px">


This notebook takes a input protein sequence of length 50-512,
And runs ESM + THOR, and writes a FASTA of optimized variants.


## 0) Runtime
Set Runtime -> Change runtime type -> GPU before running.


In [ ]:
#@title Clone repository (optional)
from pathlib import Path

GIT_REPO_URL = "https://github.com/Umbriona/THOR.git"  # TODO: set to your repo URL
REPO_DIR = Path("/content/THOR-repo")

if GIT_REPO_URL:
    if not REPO_DIR.exists():
        !git clone -q {GIT_REPO_URL} {REPO_DIR}
    else:
        print(f"Repo already exists at {REPO_DIR}")
else:
    print("Skipping clone. Set GIT_REPO_URL if you want to clone.")


In [2]:
#@title Install dependencies
%pip -q install  "transformers>=4.36" "biopython>=1.79" "pyyaml" "tqdm" "typeguard" "gdown"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.6 MB/s eta 0:00:00


In [ ]:
#@title Configure paths
import sys
from pathlib import Path

candidates = []
if "REPO_DIR" in globals() and REPO_DIR.exists():
    candidates.extend([REPO_DIR / "sandra" / "THOR", REPO_DIR ])
candidates.extend([Path("/content/THOR"), Path("/content/sandra/THOR")])

THOR_ROOT = None
for c in candidates:
    if c.exists():
        THOR_ROOT = c
        break

if THOR_ROOT is None:
    raise FileNotFoundError("Could not locate THOR. Set THOR_ROOT manually.")

sys.path.append(str(THOR_ROOT / "src"))
sys.path.append(str(THOR_ROOT / "src/scripts"))

print("THOR_ROOT:", THOR_ROOT)


THERMALGAN_ROOT: /content/thermalgan-repo


## 1) Weights
You need a run directory that contains `config.yaml` and `weights/epoch_<n>/generator_G.h5`.
Optionally, download OGT weights if you want predicted OGT values.


In [ ]:
#@title Download weights (optional) from Hugging Face
from pathlib import Path
from huggingface_hub import snapshot_download

def extract(archive, dest_dir):
    if str(archive).endswith(".zip"):
        !unzip -q {archive} -d {dest_dir}
    elif str(archive).endswith((".tar.gz", ".tgz")):
        !tar -xzf {archive} -C {dest_dir}
    else:
        print("Unknown archive extension; extract manually if needed.")


DOWNLOAD_DIR = Path("/content/THOR_weights")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

REPO_ID = "Umbriona/THOR"   # your HF repo

# Option 1: download EVERYTHING in the repo
snapshot_download(
    repo_id=REPO_ID,
    repo_type="model",                 # HF "model" repo (default)
    local_dir=str(DOWNLOAD_DIR),
    local_dir_use_symlinks=False,      # better for Colab
    resume_download=True,
)
# Extract OGT weights
arch = DOWNLOAD_DIR / "OGT.tar.gz"
extract(arch, DOWNLOAD_DIR )
print("Downloaded to:", DOWNLOAD_DIR)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Downloaded to: /content/thermalgan_weights


In [ ]:
#@title Choose model and output location
#@markdown Each THOR model is trained to hold generated variants near a
#@markdown **target sequence identity** to your input sequence. Lower values
#@markdown change more of the sequence; higher values stay closer to it.
#@markdown
#@markdown `80` is the default and matches the weights this notebook used
#@markdown before the other models were published.
from pathlib import Path

TARGET_IDENTITY = "80" #@param ["70", "75", "80", "85", "90", "95"]

DOWNLOAD_DIR = Path(globals().get("DOWNLOAD_DIR", "/content/THOR_weights"))

# Each weights_NN/ holds its own config.yaml beside the weights, so this is a
# complete run directory on its own.
RUN_DIR = DOWNLOAD_DIR / f"weights_{TARGET_IDENTITY}"

OUTPUT_DIR = Path("/content/THOR_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The ET (temperature) regressor is shared by every model, so it lives at the
# download root rather than inside a per-model directory.
OGT_WEIGHTS_DIR = DOWNLOAD_DIR
OGT_CONFIG = DOWNLOAD_DIR / "config_classifier1.yaml"

if not RUN_DIR.is_dir():
    available = sorted(p.name for p in DOWNLOAD_DIR.glob("weights_*") if p.is_dir())
    raise FileNotFoundError(
        f"No model directory at {RUN_DIR}.\n"
        f"Available: {available or 'none - run the download cell above first'}"
    )
for required in ("config.yaml", "generator_G.h5"):
    if not (RUN_DIR / required).exists():
        raise FileNotFoundError(f"Missing {required} in {RUN_DIR}")

print(f"Model      : weights_{TARGET_IDENTITY}  (target identity {TARGET_IDENTITY}%)")
print(f"Run dir    : {RUN_DIR}")
print(f"Output dir : {OUTPUT_DIR}")


## 2) Optional: structure input

Upload or fetch a backbone to add ProteinMPNN features and scoring.
Skip it to run with ESM features only.

In [ ]:
#@title (Optional) Structure input for ProteinMPNN
#@markdown Give THOR a backbone and it can blend ProteinMPNN's structure-derived
#@markdown per-position probabilities into the generator input, and score every
#@markdown variant against the fold.
#@markdown
#@markdown Leave this on `none` to run exactly as before.
#@markdown
#@markdown **The structure decides the sequence.** Positions must line up with the
#@markdown generator input, so the chain you pick here is the sequence that gets
#@markdown optimized. Clear the `sequence` box in the next cell to take it from the
#@markdown structure, or paste a sequence that matches the chain exactly.
import subprocess
from pathlib import Path

STRUCTURE_SOURCE = "none" #@param ["none", "upload", "rcsb", "alphafold"]
#@markdown RCSB entry (e.g. `4ins`) or UniProt accession for AlphaFold (e.g. `P0A6Y8`):
STRUCTURE_ACCESSION = "" #@param {type:"string"}

STRUCTURE_DIR = Path("/content/THOR_structures")
MPNN_DIR = Path("/content/ProteinMPNN")
STRUCTURE_PATH = None


def _fetch(url, dest):
    import urllib.request
    print(f"Fetching {url}")
    urllib.request.urlretrieve(url, dest)
    return dest


def _alphafold_url(accession):
    """Ask the AlphaFold API for the current model URL rather than guessing a version."""
    import json
    import urllib.request
    api = f"https://alphafold.ebi.ac.uk/api/prediction/{accession}"
    with urllib.request.urlopen(api) as fh:
        entries = json.load(fh)
    if not entries:
        raise ValueError(f"No AlphaFold model for {accession}")
    return entries[0]["pdbUrl"]


if STRUCTURE_SOURCE != "none":
    STRUCTURE_DIR.mkdir(parents=True, exist_ok=True)

    if STRUCTURE_SOURCE == "upload":
        from google.colab import files
        uploaded = files.upload()
        name = next(iter(uploaded))
        STRUCTURE_PATH = STRUCTURE_DIR / name
        STRUCTURE_PATH.write_bytes(uploaded[name])
    elif STRUCTURE_SOURCE == "rcsb":
        acc = STRUCTURE_ACCESSION.strip().lower()
        if not acc:
            raise ValueError("Set STRUCTURE_ACCESSION to a PDB entry, e.g. 4ins")
        STRUCTURE_PATH = _fetch(f"https://files.rcsb.org/download/{acc}.pdb",
                                STRUCTURE_DIR / f"{acc}.pdb")
    elif STRUCTURE_SOURCE == "alphafold":
        acc = STRUCTURE_ACCESSION.strip().upper()
        if not acc:
            raise ValueError("Set STRUCTURE_ACCESSION to a UniProt accession, e.g. P0A6Y8")
        STRUCTURE_PATH = _fetch(_alphafold_url(acc), STRUCTURE_DIR / f"AF-{acc}.pdb")

    # ProteinMPNN is a git checkout, not a pip package; the weights ship with it.
    if not (MPNN_DIR / "protein_mpnn_run.py").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/dauparas/ProteinMPNN",
                        str(MPNN_DIR)], check=True)
    print(f"ProteinMPNN: {MPNN_DIR}")

    import sys
    sys.path.append(str(THOR_ROOT / "src/scripts"))
    from data_processing.compute_mpnn_features import extract_chain, list_chains, numbering_gaps

    chains = list_chains(STRUCTURE_PATH)
    print(f"\nStructure : {STRUCTURE_PATH}")
    print(f"Chains    : {chains}")
    import tempfile
    for ch in chains:
        with tempfile.TemporaryDirectory() as tmp:
            seq, resnums = extract_chain(str(STRUCTURE_PATH), ch, Path(tmp) / "c.pdb")
        gaps = numbering_gaps(resnums)
        flag = f"  <- {len(gaps)} numbering break(s), e.g. {gaps[0][0]}->{gaps[0][1]}" if gaps else ""
        print(f"  {ch}: {len(seq)} residues{flag}")
        print(f"     {seq}")
    print("\nSet mpnn_chain in the next cell to pick a chain (default: the first one).")
else:
    print("No structure: ESM features only, exactly as before.")

In [ ]:
#@title Sequence input and generation settings
#@markdown Leave `sequence` empty to take it from the structure loaded above.
sequence = "MRKKVTIVGAGNVGATAAHWIASKELADVVLIDVVEGVPQGKGLDLLEAMPIEKRDAHILGTNDYADTANSDIVVITAGIARKPGMSRDDLLQTNYKIMSDVVGKAIAASPDCILIIVSNPLDAMAQTAYRQAKFNRNRVIGMAGVLDSARFRTFIAEELKVSVENVTAFVLGGHGDTMVPLARYSTVAGIPITELIPKERLDAIVQRTRDGGAEIVKYLKTGSAYYAPSAAAVEMVEAILKDKKKILPCAAFLEGEYGISGYYIGVPCKLGARGLEQIIEIRLTPEEEAALKKSAEAVKELCAVIGV" #@param {type:"string"}
#@markdown
sequence_id = "query1" #@param {type:"string"}
sequence_temp = None  # optional numeric temperature
output_name = sequence_id

replicates = 5 #@param {type:"integer"}
temperature = 1.0 #@param {type:"slider", min:0.1, max:1, step:0.01}


esm_init_mode = "single" #@param ["random", "single"]
esm_random_mask_prob = 0.15
esm_random_chunk = 8
esm_random_seed = 42

esm_filter_threshold = 3.0 #@param {type:"slider", min:1, max:4, step:0.05}
filter_opt_cycles = 0 #@param {type:"integer"}
sampling_cycles = 0 #@param {type:"integer"}
skip_optimize = False #@param {type:"boolean"}

#@markdown ---
#@markdown **ProteinMPNN** (ignored unless a structure was loaded above).
#@markdown
#@markdown `mpnn_weight` mixes structure into the generator's input features:
#@markdown `0` is pure ESM, which is what the model was trained on, and `1` is
#@markdown pure ProteinMPNN. Anything above 0 feeds the generator a distribution
#@markdown it never saw in training, so treat the output as experimental and
#@markdown compare against `mpnn_weight = 0` before trusting it.
mpnn_weight = 0.3 #@param {type:"slider", min:0, max:1, step:0.05}
mpnn_chain = "" #@param {type:"string"}
skip_mpnn_score = False #@param {type:"boolean"}

hf_model = "facebook/esm1v_t33_650M_UR90S_1"
esm_device = "cuda"  # cuda or cpu
batch_tokens = 20000
max_length = 1022
esm_bf16 = True
gpu_id = 0

skip_ogt = False #@param {type:"boolean"}
store_softmax = False #@param {type:"boolean"}
#@markdown ---
#@markdown The scoring models' per-position probabilities are always written:
#@markdown ESM re-scored for every variant, plus one ProteinMPNN distribution
#@markdown for the backbone whenever a structure is loaded. They are what
#@markdown `inspect_variant_scores.ipynb` reads, and the ProteinMPNN line is
#@markdown the only record of what the structure said.

In [ ]:
#@title Run THOR
import shlex
import subprocess
from subprocess import CalledProcessError

STRUCTURE_PATH = globals().get("STRUCTURE_PATH", None)
MPNN_DIR = globals().get("MPNN_DIR", None)

sequence = "".join(sequence.split())
if not sequence and not STRUCTURE_PATH:
    raise ValueError("Sequence is empty. Enter one, or load a structure to take its sequence.")

cmd = [
    "python",
    str(THOR_ROOT / "src/scripts/generate_thor.py"),
    "--run_dir", str(RUN_DIR),
    "--seq_id", sequence_id,
    "--output_dir", str(OUTPUT_DIR),
    "--name", output_name,
    "--replicates", str(replicates),
    "--temperature", str(temperature),
    "--hf_model", hf_model,
    "--device", esm_device,
    "--batch_tokens", str(batch_tokens),
    "--max_length", str(max_length),
    "--gpu", str(gpu_id),
    "--esm_init_mode", esm_init_mode,
    "--esm_random_mask_prob", str(esm_random_mask_prob),
    "--esm_random_chunk", str(esm_random_chunk),
    "--esm_random_seed", str(esm_random_seed),
    "--filter_opt_cycles", str(filter_opt_cycles),
    "--sampling_cycles", str(sampling_cycles),
]
if sequence:
    cmd += ["--sequence", sequence]
if STRUCTURE_PATH:
    cmd += [
        "--structure", str(STRUCTURE_PATH),
        "--mpnn_dir", str(MPNN_DIR),
        "--mpnn_weight", str(mpnn_weight),
    ]
    if mpnn_chain.strip():
        cmd += ["--mpnn_chain", mpnn_chain.strip()]
    if skip_mpnn_score:
        cmd += ["--skip_mpnn_score"]
if sequence_temp is not None:
    cmd += ["--seq_temp", str(sequence_temp)]
if store_softmax:
    cmd += ["--store_softmax"]
# Always: one ProteinMPNN line for the backbone and one ESM line per variant.
cmd += ["--store_scoring_probs"]
if esm_filter_threshold is not None:
    cmd += ["--esm_filter_threshold", str(esm_filter_threshold)]
if skip_optimize:
    cmd += ["--skip_optimize"]
if esm_bf16:
    cmd += ["--esm_bf16"]
if skip_ogt:
    cmd += ["--skip_ogt"]
else:
    cmd += ["--ogt_config", str(OGT_CONFIG)]
    cmd += [
        "--ogt_weights",
        str(DOWNLOAD_DIR / "OGT" / "Model1" / "model_weights.h5"),
        str(DOWNLOAD_DIR / "OGT" / "Model2" / "model_weights.h5"),
        str(DOWNLOAD_DIR / "OGT" / "Model3" / "model_weights.h5"),
    ]

print("Running:", " ".join(shlex.quote(c) for c in cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print("Return Code:", result.returncode)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)

if result.returncode != 0:
    raise CalledProcessError(result.returncode, cmd, output=result.stdout, stderr=result.stderr)

In [ ]:
#@title Inspect outputs
from pathlib import Path

fasta_path = OUTPUT_DIR / f"{output_name}_variants.fasta"
print("Output FASTA:", fasta_path)

if fasta_path.exists():
    !head -n 20 {fasta_path}
else:
    print("FASTA not found; check logs above.")

print("\nFiles written so far:")
for p in sorted(OUTPUT_DIR.glob(f"{output_name}_*")):
    print(f"  {p.name}  ({p.stat().st_size/1024:.1f} KB)")
print("\nEverything is bundled into one zip at the end of the notebook.")

## 3) Visualise the variants

Parses the variant FASTA into a table, then plots the predicted temperature
shift against the wildtype and the per-variant scores.

In [ ]:
#@title Parse the variant FASTA into a table
import pandas as pd

NUMERIC_FIELDS = [
    "predicted_OGT", "G_global_nll", "G_sub_nll",
    "ESM_global_nll", "ESM_sub_nll", "Identity_to_WT(%)", "temp",
    "filter_threshold", "MPNN_nll",
]


def parse_variant_fasta(path):
    """Read the FASTA written by generate_thor.py into a DataFrame.

    Headers look like:
    >label | type=sampled | predicted_OGT=67.3 | ... | reverted_pos=M1L,M50V
    """
    rows, rec, seq = [], None, []

    def flush():
        if rec is not None:
            rec["seq"] = "".join(seq)
            rec["length"] = len(rec["seq"])
            rows.append(rec)

    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                flush()
                parts = [p.strip() for p in line[1:].split("|")]
                rec, seq = {"label": parts[0]}, []
                for part in parts[1:]:
                    if "=" in part:
                        key, val = part.split("=", 1)
                        rec[key.strip()] = val.strip()
            else:
                seq.append(line)
    flush()

    df = pd.DataFrame(rows)
    for col in NUMERIC_FIELDS:
        if col in df:
            df[col] = pd.to_numeric(df[col].replace("NA", pd.NA), errors="coerce")
    if "reverted_pos" in df:
        df["n_reverted"] = df["reverted_pos"].fillna("").apply(
            lambda s: len([m for m in s.split(",") if m])
        )
    df = df.rename(columns={"type": "kind", "Identity_to_WT(%)": "identity_to_wt"})

    # Predicted temperature shift relative to the wildtype of the same input.
    wt_ogt = df.loc[df["kind"] == "wt", "predicted_OGT"]
    df["delta_T"] = df["predicted_OGT"] - (wt_ogt.iloc[0] if len(wt_ogt) else float("nan"))

    # ProteinMPNN scores sit far from zero and close together, so the useful
    # quantity is the shift against the wildtype on the same backbone.
    if "MPNN_nll" in df:
        wt_mpnn = df.loc[df["kind"] == "wt", "MPNN_nll"]
        df["delta_MPNN"] = df["MPNN_nll"] - (wt_mpnn.iloc[0] if len(wt_mpnn) else float("nan"))
    return df


fasta_path = OUTPUT_DIR / f"{output_name}_variants.fasta"
variants = parse_variant_fasta(fasta_path)
TABLE_PATH = OUTPUT_DIR / f"{output_name}_variants.csv"
variants.to_csv(TABLE_PATH, index=False)
print(f"{len(variants)} sequences  |  kinds: {variants['kind'].value_counts().to_dict()}")
print("Table:", TABLE_PATH)
variants.drop(columns=["seq"]).head(50)

In [ ]:
#@title Plot predicted temperature shift and sequence scores
import matplotlib.pyplot as plt

PLOT_PATH = OUTPUT_DIR / f"{output_name}_variants.png"

SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_SOFT = "#52514e"
GRID = "#e3e2de"
WARMER = "#eb6834"   # positive dT
COOLER = "#2a78d6"   # negative dT
SCORE_1 = "#2a78d6"  # ESM
SCORE_2 = "#1baf7a"  # generator
SCORE_3 = "#6250d6"  # ProteinMPNN


def short_label(label):
    """query1_variant_rep0_filtered -> rep0 filtered"""
    parts = label.split("_")
    if "variant" in parts:
        parts = parts[parts.index("variant") + 1:]
    elif len(parts) > 1:
        parts = parts[1:]
    return " ".join(parts)


def plot_variants(df, sort_by="delta_T"):
    plot_df = df[df["kind"] != "wt"].copy()
    if plot_df.empty:
        print("No variants to plot.")
        return
    plot_df = plot_df.sort_values(sort_by, ascending=False)  # best on top (y axis inverted)
    y = range(len(plot_df))
    labels = [short_label(l) for l in plot_df["label"]]

    panels = [
        ("delta_T", "Predicted ΔT vs wildtype  (°C)", None),
        ("ESM_sub_nll", "ESM NLL at substitutions  (lower = more plausible)", SCORE_1),
        ("G_sub_nll", "Generator NLL at substitutions  (lower = more likely)", SCORE_2),
        ("delta_MPNN", "ProteinMPNN NLL vs wildtype  (negative = fits the fold better)", SCORE_3),
    ]
    panels = [p for p in panels if p[0] in plot_df and plot_df[p[0]].notna().any()]

    fig, axes = plt.subplots(
        1, len(panels), sharey=True,
        figsize=(4.3 * len(panels), 0.42 * len(plot_df) + 2.2),
        facecolor=SURFACE,
    )
    axes = [axes] if len(panels) == 1 else list(axes)

    for ax, (col, title, color) in zip(axes, panels):
        vals = plot_df[col].fillna(0.0).to_numpy()
        colors = [WARMER if v >= 0 else COOLER for v in vals] if color is None else color
        ax.barh(list(y), vals, height=0.62, color=colors, zorder=3)
        ax.set_title(title, fontsize=10.5, color=INK, loc="left", pad=10)
        ax.set_facecolor(SURFACE)
        ax.xaxis.grid(True, color=GRID, lw=0.8, zorder=0)
        ax.set_axisbelow(True)
        for side in ("top", "right", "left"):
            ax.spines[side].set_visible(False)
        ax.spines["bottom"].set_color(GRID)
        ax.tick_params(colors=INK_SOFT, labelsize=9, length=0)
        if color is None or vals.min() < 0:
            ax.axvline(0, color=INK_SOFT, lw=1.0, zorder=4)

        span = max(abs(vals.max()), abs(vals.min())) or 1.0
        pad = span * 0.03
        for yi, v in zip(y, vals):
            ax.text(
                v + (pad if v >= 0 else -pad), yi, f"{v:,.1f}" if col == "delta_T" else f"{v:,.2f}",
                va="center", ha="left" if v >= 0 else "right",
                fontsize=8.5, color=INK_SOFT,
            )
        lo, hi = min(0, vals.min()), max(0, vals.max())
        ax.set_xlim(lo - span * 0.18, hi + span * 0.18)

    axes[0].set_yticks(list(y))
    axes[0].set_yticklabels(labels, fontsize=9, color=INK)
    axes[0].invert_yaxis()

    wt_ogt = df.loc[df["kind"] == "wt", "predicted_OGT"]
    wt_txt = f"wildtype predicted OGT {wt_ogt.iloc[0]:.1f} °C" if len(wt_ogt) else "no wildtype record"
    fig.suptitle(
        f"THOR variants for {sequence_id}  ({wt_txt})",
        fontsize=12.5, color=INK, x=0.012, ha="left", y=0.995,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    fig.savefig(PLOT_PATH, dpi=150, facecolor=SURFACE, bbox_inches="tight")
    print("Figure:", PLOT_PATH)
    plt.show()


plot_variants(variants)

## 3b) Residue-level scores

Why a variant scored the way it did, position by position: the distribution each
scoring model puts over the 20 amino acids, with the residue actually present
boxed, the wildtype residue marked, and a track above showing how much
probability the residue present got. A short bar under a high wildtype mark is a
substitution the model dislikes.

Rows are grouped by side-chain chemistry. Whole sequences do not fit on a screen,
so the view is windowed - or switch on **substitutions only** to see just the
positions that changed.


In [ ]:
#@title Load the per-position probabilities
import json
import numpy as np

AA_ORDER = "ACDEFGHIKLMNPQRSTVWY"   # the column order THOR writes

RECORDS = variants.to_dict("records")
BY_LABEL = {r["label"]: r for r in RECORDS}
WT = next((r for r in RECORDS if r.get("kind") == "wt"), RECORDS[0])
WT_SEQ = WT["seq"]

# ESM is written per emitted sequence; ProteinMPNN once for the backbone. The
# key is "Scoring_Model" - both models are also used as feature models, and this
# file only ever holds the scoring side. Older runs wrote "model".
ESM, MPNN, GEN = {}, None, {}
for f in OUTPUT_DIR.glob(f"{output_name}_scoring_probs.jsonl"):
    for line in f.read_text().splitlines():
        if not line.strip():
            continue
        d = json.loads(line)
        probs = np.asarray(d["probs"], dtype=np.float32)
        which = d.get("Scoring_Model", d.get("model"))
        if which == "esm":
            ESM[d["label"]] = probs
        elif which == "proteinmpnn":
            MPNN = probs

# The generator softmax has 21 columns; the last one is not an amino acid.
for f in OUTPUT_DIR.glob(f"{output_name}_variants_softmax.jsonl"):
    for line in f.read_text().splitlines():
        if not line.strip():
            continue
        d = json.loads(line)
        GEN[d["id"].rsplit("_rep", 1)[0]] = np.asarray(d["probs"], dtype=np.float32)[:, :20]

print(f"{len(RECORDS)} sequences  (wildtype: {WT['label']}, {len(WT_SEQ)} aa)")
print(f"ESM distributions     : {len(ESM)}")
print(f"ProteinMPNN           : {'yes, ' + str(len(MPNN)) + ' positions' if MPNN is not None else 'none  (load a structure in section 2)'}")
print(f"Generator softmax     : {len(GEN)}" if GEN else
      "Generator softmax     : none  (tick store_softmax and re-run)")


def models_for(label):
    """(name, probs) for every score available for this variant."""
    out = []
    if label in ESM:
        out.append(("ESM", ESM[label]))
    for key, probs in GEN.items():
        if label.startswith(key):
            out.append(("Generator", probs))
            break
    if MPNN is not None:
        out.append(("ProteinMPNN", MPNN))
    return out


def substitutions(label):
    """Positions (0-based) where this variant differs from the wildtype."""
    return [i for i, (a, b) in enumerate(zip(BY_LABEL[label]["seq"], WT_SEQ)) if a != b]


In [ ]:
#@title Grouping, colours and the plotting function
# Kept identical to scripts/inspect_variant_scores.ipynb.
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

AA_GROUPS = [
    ("hydrophobic", "AVLIM"),
    ("aromatic", "FWY"),
    ("polar", "STNQ"),
    ("acidic", "DE"),
    ("basic", "KRH"),
    ("special", "CGP"),
]
GROUPED_AA = "".join(aa for _, aa in AA_GROUPS)
ROW_OF = {aa: i for i, aa in enumerate(GROUPED_AA)}
REORDER = [AA_ORDER.index(aa) for aa in GROUPED_AA]
assert sorted(GROUPED_AA) == sorted(AA_ORDER), "the grouping must cover all 20 residues"

SURFACE, INK, INK_SOFT = "#fcfcfb", "#0b0b0b", "#52514e"
GRID, RULE, WARMER, MUTED = "#e3e2de", "#c9c7c2", "#eb6834", "#9a9894"
MODEL_HUE = {"ESM": "#2a78d6", "Generator": "#1baf7a", "ProteinMPNN": "#6250d6"}

# Colour and height limits come from the whole sequence, not the window, so that
# scrolling does not silently rescale the picture under you.
_SCALE = {}


def scale_for(name, probs):
    if name not in _SCALE:
        _SCALE[name] = (float(np.quantile(probs, 0.995)), float(probs.max()))
    return _SCALE[name]


def plot_window(probs, var_seq, cols, model_name, variant_label, contiguous=True):
    """One figure for one score. `cols` are the 0-based positions to draw."""
    cols = list(cols)
    n = len(cols)
    var_win = "".join(var_seq[i] for i in cols)
    wt_win = "".join(WT_SEQ[i] for i in cols)
    win = probs[cols]
    grid = win[:, REORDER].T
    hue = MODEL_HUE.get(model_name, "#2a78d6")
    vmax, track_max = scale_for(model_name, probs)
    cmap = LinearSegmentedColormap.from_list("ramp", [SURFACE, hue])

    p_var = win[np.arange(n), [AA_ORDER.index(a) for a in var_win]]
    p_wt = win[np.arange(n), [AA_ORDER.index(a) for a in wt_win]]
    is_sub = np.array([a != b for a, b in zip(var_win, wt_win)])

    # Scattered columns get wider cells, since there are far fewer of them.
    fig_w = max(9.5, n * 0.215) if contiguous else max(8.0, n * 0.55)
    fig = plt.figure(figsize=(fig_w, 7.6), facecolor=SURFACE)
    # Margins are fixed in inches, so the group labels and the colourbar keep
    # their room whatever the figure width works out to.
    gs = fig.add_gridspec(2, 1, height_ratios=[1, 2.7], hspace=0.07,
                          left=1.57 / fig_w, right=1 - 0.95 / fig_w,
                          top=0.855, bottom=0.085)
    ax_t = fig.add_subplot(gs[0])
    ax_h = fig.add_subplot(gs[1], sharex=ax_t)

    # --- how likely is the residue that is actually there -------------------
    ax_t.set_facecolor(SURFACE)
    if contiguous:
        for i in np.flatnonzero(is_sub):
            ax_t.axvspan(i - 0.5, i + 0.5, color=WARMER, alpha=0.09, lw=0, zorder=0)
    ax_t.bar(range(n), p_var, width=0.72,
             color=[WARMER if s else MUTED for s in is_sub], zorder=3)
    if is_sub.any():
        sub_i = np.flatnonzero(is_sub)
        ax_t.scatter(sub_i, p_wt[sub_i], s=34, marker="_", color=INK,
                     linewidths=1.7, zorder=5)
        for i in sub_i:
            ax_t.annotate(f"{wt_win[i]}{cols[i] + 1}{var_win[i]}",
                          (i, max(p_var[i], p_wt[i])), textcoords="offset points",
                          xytext=(0, 6), ha="center", fontsize=7.5, color=WARMER,
                          rotation=90, zorder=6)
    ax_t.set_ylim(0, track_max * 1.28)
    ax_t.set_ylabel("probability of\nthe residue present", fontsize=8.5, color=INK_SOFT)
    ax_t.yaxis.grid(True, color=GRID, lw=0.8)
    ax_t.set_axisbelow(True)
    for s in ("top", "right", "left"):
        ax_t.spines[s].set_visible(False)
    ax_t.spines["bottom"].set_color(GRID)
    ax_t.tick_params(colors=INK_SOFT, labelsize=8, length=0, labelbottom=False)

    # --- the full distribution over all 20 residues -------------------------
    im = ax_h.imshow(grid, aspect="auto", cmap=cmap, vmin=0, vmax=vmax,
                     interpolation="nearest", extent=(-0.5, n - 0.5, 19.5, -0.5))
    ax_h.set_yticks(range(20))
    ax_h.set_yticklabels(list(GROUPED_AA), fontsize=8.5, color=INK, family="monospace")

    row = 0
    for name, members in AA_GROUPS:
        if row:
            ax_h.axhline(row - 0.5, color=RULE, lw=1.2, zorder=5)
        ax_h.text(-0.085, row + len(members) / 2 - 0.5, name, fontsize=8, color=INK_SOFT,
                  va="center", ha="right", transform=ax_h.get_yaxis_transform())
        row += len(members)

    for i, (a, b) in enumerate(zip(var_win, wt_win)):
        on_dark = grid[ROW_OF[a], i] > 0.55 * vmax
        ax_h.add_patch(Rectangle(
            (i - 0.5, ROW_OF[a] - 0.5), 1, 1, fill=False,
            ec=WARMER if a != b else (SURFACE if on_dark else INK),
            lw=1.8 if a != b else 1.0, zorder=6))
        if a != b:  # where the substitution came from
            ax_h.add_patch(Rectangle((i - 0.5, ROW_OF[b] - 0.5), 1, 1, fill=False,
                                     ec=INK_SOFT, lw=1.1, ls=(0, (1.4, 1.2)), zorder=6))

    ax_h.set_xlim(-0.5, n - 0.5)
    ax_h.set_xticks(range(n))
    ax_h.set_xticklabels(list(var_win), fontsize=7.5, family="monospace")
    for i, tick in enumerate(ax_h.get_xticklabels()):
        tick.set_color(WARMER if is_sub[i] else INK)
    if contiguous:
        step = 10 if n > 40 else 5
        label_at = [i for i in range(n) if (cols[i] + 1) % step == 0]
    else:
        label_at = range(n)  # the columns are not adjacent, so number every one
    for i in label_at:
        ax_h.annotate(str(cols[i] + 1), (i, -0.055),
                      xycoords=ax_h.get_xaxis_transform(), ha="center", va="top",
                      fontsize=7, color=WARMER if is_sub[i] else INK_SOFT,
                      rotation=0 if contiguous else 90)
    ax_h.tick_params(colors=INK_SOFT, length=0, pad=2)
    for s in ax_h.spines.values():
        s.set_visible(False)

    cax = fig.add_axes([1 - 0.78 / fig_w, 0.085, 0.115 / fig_w, 0.54])
    cb = fig.colorbar(im, cax=cax)
    cb.set_label(f"{model_name} probability", fontsize=8, color=INK_SOFT)
    cb.outline.set_visible(False)
    cb.ax.tick_params(colors=INK_SOFT, labelsize=7.5, length=0)

    handles = [
        Rectangle((0, 0), 1, 1, fill=False, ec=INK, lw=1.0),
        Rectangle((0, 0), 1, 1, fill=False, ec=INK_SOFT, lw=1.1, ls=(0, (1.4, 1.2))),
        Rectangle((0, 0), 1, 1, fill=False, ec=WARMER, lw=1.8),
        Line2D([0], [0], marker="_", color=INK, lw=0, markersize=9, markeredgewidth=1.7),
    ]
    fig.legend(handles, ["residue in this variant", "wildtype residue", "substitution",
                         "p(wildtype) at a substitution"],
               loc="upper right", bbox_to_anchor=(1 - 0.45 / fig_w, 0.945),
               ncol=4 if fig_w > 9 else 2, frameon=False,
               fontsize=8, labelcolor=INK_SOFT, handletextpad=0.5, columnspacing=1.5)

    n_sub = int(is_sub.sum())
    where = (f"residues {cols[0] + 1}-{cols[-1] + 1}" if contiguous
             else f"{n} substitution{'s' if n != 1 else ''} only")
    fig.suptitle(
        f"{variant_label}   ·   {model_name}   ·   {where}"
        + (f"   ({n_sub} substitution{'s' if n_sub != 1 else ''} in view)" if contiguous else ""),
        fontsize=11.5, color=INK, x=0.012, ha="left", y=0.985)
    return fig


def show(variant_label, start=0, width=50, models=None, subs_only=False):
    """One figure per score. `subs_only` drops every position that did not change."""
    var_seq = BY_LABEL[variant_label]["seq"]
    available = models_for(variant_label)
    if models:
        available = [(n, p) for n, p in available if n in models]
    if not available:
        print("No probability files for this variant - see the load cell above.")
        return

    if subs_only:
        cols = substitutions(variant_label)
        if not cols:
            print(f"{variant_label} is identical to the wildtype - nothing to show.")
            return
    else:
        cols = range(start, min(start + width, len(var_seq)))

    for name, probs in available:
        if len(probs) < len(var_seq):
            print(f"[warn] {name} covers {len(probs)} of {len(var_seq)} positions; skipping")
            continue
        plot_window(probs, var_seq, cols, name, variant_label, contiguous=not subs_only)
        plt.show()


print("plotting ready")

In [ ]:
#@title Interactive viewer
try:
    import ipywidgets as W
    from IPython.display import display
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False

VARIANTS = [r["label"] for r in RECORDS]
DEFAULT = next((v for v in VARIANTS if BY_LABEL[v].get("kind") != "wt"), VARIANTS[0])

if HAVE_WIDGETS:
    w_variant = W.Dropdown(options=VARIANTS, value=DEFAULT, description="Variant:",
                           layout=W.Layout(width="420px"))
    w_width = W.Dropdown(options=[30, 40, 50, 60, 80, 100], value=50,
                         description="Window:", layout=W.Layout(width="200px"))
    w_start = W.IntSlider(min=0, max=max(0, len(WT_SEQ) - 50), step=5, value=0,
                          description="Start:", continuous_update=False,
                          layout=W.Layout(width="620px"))
    w_models = W.SelectMultiple(options=[n for n, _ in models_for(DEFAULT)] or ["(none)"],
                                value=tuple(n for n, _ in models_for(DEFAULT)),
                                description="Scores:", rows=3,
                                layout=W.Layout(width="260px"))
    w_jump = W.Dropdown(options=[("(jump to a substitution)", -1)], value=-1,
                        description="Jump to:", layout=W.Layout(width="420px"))
    w_prev, w_next = W.Button(description="◀", layout=W.Layout(width="45px")), \
                     W.Button(description="▶", layout=W.Layout(width="45px"))
    w_subs = W.Checkbox(value=False, description="substitutions only", indent=False,
                        layout=W.Layout(width="200px"))

    def refresh_variant(*_):
        """Window bounds and the jump list belong to the selected variant."""
        label = w_variant.value
        w_start.max = max(0, len(BY_LABEL[label]["seq"]) - w_width.value)
        w_start.step = max(1, w_width.value // 5)
        names = [n for n, _ in models_for(label)]
        w_models.options = names or ["(none)"]
        w_models.value = tuple(names)
        var = BY_LABEL[label]["seq"]
        opts = [("(jump to a substitution)", -1)]
        opts += [(f"{WT_SEQ[i]}{i+1}{var[i]}", i) for i in substitutions(label)]
        w_jump.options = opts
        w_jump.value = -1
        set_scroll_enabled()

    def set_scroll_enabled(*_):
        """Scrolling means nothing when only the substituted columns are drawn."""
        off = w_subs.value
        for w in (w_start, w_width, w_jump, w_prev, w_next):
            w.disabled = off

    def do_jump(change):
        if change["new"] is not None and change["new"] >= 0:
            w_start.value = max(0, min(w_start.max, change["new"] - w_width.value // 2))

    def step(delta):
        return lambda _: setattr(w_start, "value",
                                 max(0, min(w_start.max, w_start.value + delta * w_width.value)))

    w_variant.observe(refresh_variant, names="value")
    w_width.observe(refresh_variant, names="value")
    w_subs.observe(set_scroll_enabled, names="value")
    w_jump.observe(do_jump, names="value")
    w_prev.on_click(step(-1))
    w_next.on_click(step(+1))
    refresh_variant()

    out = W.interactive_output(
        lambda variant, start, width, models, subs_only:
            show(variant, start, width, list(models), subs_only),
        {"variant": w_variant, "start": w_start, "width": w_width,
         "models": w_models, "subs_only": w_subs},
    )
    display(W.VBox([
        W.HBox([w_variant, w_width, w_subs]),
        W.HBox([w_jump, w_models]),
        W.HBox([w_prev, w_start, w_next]),
        out,
    ]))
else:
    print("ipywidgets missing - calling show() directly instead.")
    show(DEFAULT, subs_only=True)

In [ ]:
#@title (Optional) Save residue figures into the bundle
#@markdown Every window of every score of every variant runs to hundreds of
#@markdown megabytes, so this saves a deliberately small selection: the best
#@markdown variants by predicted ΔT, one figure per score.
SAVE_TOP_N = 5 #@param {type:"integer"}
#@markdown Only the positions that changed. Untick to save the whole sequence
#@markdown in windows instead, which is roughly ten times as many figures.
SAVE_SUBSTITUTIONS_ONLY = True #@param {type:"boolean"}
SAVE_WINDOW = 50 #@param {type:"integer"}

FIG_DIR = OUTPUT_DIR / f"{output_name}_figures"

ranked = variants[variants["kind"] != "wt"].sort_values("delta_T", ascending=False)
chosen = list(ranked["label"].head(max(0, SAVE_TOP_N)))
written = []

for label in chosen:
    var_seq = BY_LABEL[label]["seq"]
    subs = substitutions(label)
    if not subs:
        continue
    if SAVE_SUBSTITUTIONS_ONLY:
        windows = [(subs, False)]
    else:
        # One window per substitution, skipping any already visible in the last.
        starts, covered_to = [], 0
        for i in subs:
            if starts and i < covered_to:
                continue
            s = max(0, min(len(var_seq) - SAVE_WINDOW, i - SAVE_WINDOW // 2))
            starts.append(s)
            covered_to = s + SAVE_WINDOW
        windows = [(range(s, min(s + SAVE_WINDOW, len(var_seq))), True) for s in starts]

    for cols, contiguous in windows:
        cols = list(cols)
        for name, probs in models_for(label):
            if len(probs) < len(var_seq):
                continue
            FIG_DIR.mkdir(parents=True, exist_ok=True)
            fig = plot_window(probs, var_seq, cols, name, label, contiguous=contiguous)
            tag = f"{cols[0] + 1}-{cols[-1] + 1}" if contiguous else "substitutions"
            path = FIG_DIR / f"{label}_{name}_{tag}.png"
            fig.savefig(path, dpi=150, facecolor=SURFACE)
            plt.close(fig)
            written.append(path)

if written:
    size = sum(f.stat().st_size for f in written) / 1024
    print(f"{len(written)} figure(s), {size:,.0f} KB, in {FIG_DIR}")
    print("The bundle cell picks them up under figures/.")
else:
    print("No figures written - set SAVE_TOP_N above 0, or check the variants have substitutions.")


## 4) Download everything

One zip holding the FASTA, the table, the figure, the scoring probabilities,
the structure that was used, and a `run_metadata.json` listing every setting.


In [ ]:
#@title Bundle outputs and download
import json
import shutil
import zipfile
from pathlib import Path

bundle_path = OUTPUT_DIR / f"{output_name}_thor_outputs.zip"
STRUCTURE_PATH = globals().get("STRUCTURE_PATH", None)

# The backbone travels with the scores it produced: without it the ProteinMPNN
# line in the probabilities file cannot be checked, recomputed or reused.
structure_copy = None
if STRUCTURE_PATH:
    structure_copy = OUTPUT_DIR / f"{output_name}_structure{Path(STRUCTURE_PATH).suffix}"
    if Path(STRUCTURE_PATH).resolve() != structure_copy.resolve():
        shutil.copyfile(STRUCTURE_PATH, structure_copy)

# generate_thor.py recorded every setting it was handed; add what only the
# notebook knows, so one file describes the whole run.
meta_path = OUTPUT_DIR / f"{output_name}_run_metadata.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    meta["colab"] = {
        "notebook": "generate_thor_colab.ipynb",
        "target_identity_percent": int(TARGET_IDENTITY),
        "sequence_id": sequence_id,
        "sequence": sequence,
        "structure_source": globals().get("STRUCTURE_SOURCE", "none"),
        "structure_accession": globals().get("STRUCTURE_ACCESSION", "") or None,
        "structure_in_bundle": structure_copy.name if structure_copy else None,
    }
    meta_path.write_text(json.dumps(meta, indent=2) + "\n")
else:
    print("[warn] No run metadata written - pull the latest THOR and re-run the generation cell.")

# Everything this run wrote, except earlier bundles.
payload = sorted(
    p for p in OUTPUT_DIR.glob(f"{output_name}_*")
    if p.is_file() and p.suffix != ".zip"
)
if not payload:
    raise FileNotFoundError(f"Nothing to bundle in {OUTPUT_DIR} - run the cells above first.")

# Residue figures, if the optional cell above wrote any.
fig_dir = OUTPUT_DIR / f"{output_name}_figures"
figures = sorted(fig_dir.glob("*.png")) if fig_dir.is_dir() else []

with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in payload:
        zf.write(p, p.name)
    for p in figures:
        zf.write(p, f"figures/{p.name}")

print(f"{bundle_path.name}  ({bundle_path.stat().st_size/1024:.1f} KB) contains:")
for p in payload:
    print(f"  {p.name}  ({p.stat().st_size/1024:.1f} KB)")
if figures:
    print(f"  figures/  ({len(figures)} png, "
          f"{sum(f.stat().st_size for f in figures)/1024:,.0f} KB)")

try:
    from google.colab import files
    files.download(str(bundle_path))
except Exception as exc:
    print(f"\nDownload skipped: {exc}")
